In [2]:
# =============================================================================
# BLOCK 1: SETUP, IMPORTS, AND DATA LOADING
# =============================================================================
import warnings
warnings.filterwarnings('ignore')
import time
# --- Library Imports ---
import pandas as pd
import numpy as np
import gc
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import catboost as cb
import optuna
print("Libraries imported successfully.")
# --- Helper Function for Winkler Score ---
def winkler_score(y_true, lower, upper, alpha=0.1, return_coverage=False):
    width = upper - lower
    penalty_lower = np.where(y_true < lower, (2 / alpha) * (lower - y_true), 0)
    penalty_upper = np.where(y_true > upper, (2 / alpha) * (y_true - upper), 0)
    score = width + penalty_lower + penalty_upper
    if return_coverage:
        coverage = np.mean((y_true >= lower) & (y_true <= upper))
        return np.mean(score), coverage
    return np.mean(score)
# --- Global Constants ---
N_SPLITS = 5
RANDOM_STATE = 42
DATA_PATH = './'
N_OPTUNA_TRIALS = 30 # A strong number for a comprehensive search
COMPETITION_ALPHA = 0.1

# --- Load Raw Data ---
try:
    # We drop the low-variance columns they identified right away
    drop_cols=['id', 'golf', 'view_rainier', 'view_skyline', 'view_lakesamm','view_otherwater', 'view_other']
    df_train = pd.read_csv(DATA_PATH + 'dataset.csv').drop(columns=drop_cols)
    df_test = pd.read_csv(DATA_PATH + 'test.csv').drop(columns=drop_cols)
    print("Raw data loaded successfully.")
except FileNotFoundError:
    print("ERROR: Could not find 'dataset.csv' or 'test.csv'.")
    exit()
# --- Prepare Target Variable ---
y_true = df_train['sale_price'].copy()
# The mean-error model works best when predicting the raw price directly
# So, we will NOT log-transform the target this time.
# df_train.drop('sale_price', axis=1, inplace=True) # We keep sale_price for FE
print("Setup complete.")


Libraries imported successfully.
Raw data loaded successfully.
Setup complete.


In [5]:
# =============================================================================
# BLOCK 2: SYNTHESIZED FEATURE ENGINEERING (CORRECTED)
# =============================================================================
print("--- Starting Block 2: Synthesized Feature Engineering ---")
def create_synthesized_features(df_train, df_test):
    # Combine for consistent processing and reset the index
    df_train['is_train'] = 1
    df_test['is_train'] = 0
    # Store the original id for later, as reset_index will remove it
    train_ids = df_train.index
    test_ids = df_test.index
    all_data = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)
    
    # --- A) Brute-Force Numerical Interactions ---
    print("Creating brute-force numerical interaction features...")
    NUMS = ['area', 'land_val', 'imp_val', 'sqft_lot', 'sqft', 'sqft_1','grade', 'year_built']
    for i in range(len(NUMS)):
        for j in range(i + 1, len(NUMS)):
            all_data[f'{NUMS[i]}_x_{NUMS[j]}'] = all_data[NUMS[i]] *all_data[NUMS[j]]
    
    # --- B) Date Features ---
    all_data['sale_date'] = pd.to_datetime(all_data['sale_date'])
    all_data['year'] = all_data['sale_date'].dt.year
    all_data['month'] = all_data['sale_date'].dt.month
    all_data['year_diff'] = all_data['year'] - all_data['year_built']
    
    # --- C) TF-IDF Text Features ---
    print("Creating TF-IDF features for text columns...")
    text_cols = ['subdivision', 'zoning', 'city', 'sale_warning','join_status', 'submarket']
    all_data[text_cols] = all_data[text_cols].fillna('missing').astype(str)
    for col in text_cols:
        tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3, 5),max_features=128, binary=True)
        tfidf_matrix = tfidf.fit_transform(all_data[col])
        svd = TruncatedSVD(n_components=8, random_state=RANDOM_STATE)
        tfidf_svd = svd.fit_transform(tfidf_matrix)
        tfidf_df = pd.DataFrame(tfidf_svd, columns=[f'{col}_tfidf_svd_{i}' for i in range(8)])
        
        # This concat will now work because both have a simple 0-based index
        all_data = pd.concat([all_data, tfidf_df], axis=1)
    
    # --- D) Log transform some of the new interaction features ---
    for c in ['land_val_x_imp_val', 'land_val_x_sqft', 'imp_val_x_sqft']:
        if c in all_data.columns:
            # Add a small constant to avoid log(0)
            all_data[c] = np.log1p(all_data[c].fillna(0))
    
    # --- E) Final Cleanup ---
    print("Finalizing feature set...")
    cols_to_drop = ['sale_date', 'subdivision', 'zoning', 'city','sale_warning', 'join_status', 'submarket']
    all_data = all_data.drop(columns=cols_to_drop)
    all_data.fillna(0, inplace=True)
    
    # Separate final datasets
    X = all_data[all_data['is_train'] == 1].drop(columns=['is_train','sale_price'])
    X_test = all_data[all_data['is_train'] == 0].drop(columns=['is_train','sale_price'])
    
    # Restore the original 'id' as the index
    X.index = train_ids
    X_test.index = test_ids
    X_test = X_test[X.columns]
    return X, X_test

# We need to re-run this from the original dataframes
X, X_test = create_synthesized_features(df_train, df_test)
print(f"\nSynthesized FE complete. Total features: {X.shape[1]}")
gc.collect()


--- Starting Block 2: Synthesized Feature Engineering ---
Creating brute-force numerical interaction features...
Creating TF-IDF features for text columns...
Finalizing feature set...

Synthesized FE complete. Total features: 111


2280

In [7]:
# =======================================================================================
#
# BLOCK 3: LOAD ALL PRE-TRAINED MODEL PREDICTIONS
#
# =======================================================================================
PREDS_PATH = './model_predictions/'
print("Loading all base model predictions from saved .npy files...")

# --- Load XGBoost Predictions ---
# In your training script, these were oof_xgb_preds and test_mean_preds
oof_xgb_preds = np.load(f'{PREDS_PATH}oof_xgb_preds.npy') 
test_xgb_preds = np.load(f'{PREDS_PATH}test_xgb_preds.npy')
print("  - XGBoost predictions loaded.")

# --- Load CatBoost Predictions ---
# In your training script, these were oof_cb_preds and test_catboost_preds
oof_cb_preds = np.load(f'{PREDS_PATH}oof_cb_preds.npy')
test_cb_preds = np.load(f'{PREDS_PATH}test_cb_preds.npy') # Ensure this matches the saved file name
print("  - CatBoost predictions loaded.")

# --- Load Neural Network Predictions ---
oof_nn_preds = np.load(f'{PREDS_PATH}oof_nn_preds.npy')
test_nn_preds = np.load(f'{PREDS_PATH}test_nn_preds.npy')
print("  - Neural Network predictions loaded.")
    
y_true = pd.read_csv(DATA_PATH + 'dataset.csv')['sale_price']
print("\nAll predictions loaded successfully.")

# --- OOF RMSE of Loaded Models ---
print("\n--- OOF RMSE of All Base Models ---")
print(f"XGBoost  : ${np.sqrt(mean_squared_error(y_true, oof_xgb_preds)):,.2f}")
print(f"CatBoost : ${np.sqrt(mean_squared_error(y_true, oof_cb_preds)):,.2f}")
print(f"NeuralNet: ${np.sqrt(mean_squared_error(y_true, oof_nn_preds)):,.2f}")

Loading all base model predictions from saved .npy files...
  - XGBoost predictions loaded.
  - CatBoost predictions loaded.
  - Neural Network predictions loaded.

All predictions loaded successfully.

--- OOF RMSE of All Base Models ---
XGBoost  : $98,990.27
CatBoost : $98,246.71
NeuralNet: $108,435.86


In [10]:
# =======================================================================================
#
# BLOCK 4: FIND THE OPTIMAL 3-MODEL BLEND
#
# =======================================================================================
from scipy.optimize import minimize

print("\n--- Searching for the optimal 3-model blend weights ---")

# Stack the OOF predictions into a single array for the optimizer
oof_preds_stack = np.vstack([oof_xgb_preds, oof_cb_preds, oof_nn_preds]).T

def get_ensemble_rmse(weights):
    final_prediction = np.dot(oof_preds_stack, weights)
    return np.sqrt(mean_squared_error(y_true, final_prediction))

initial_weights = [1/3, 1/3, 1/3]
constraints = ({'type': 'eq', 'fun': lambda w: 1 - np.sum(w)})
bounds = [(0, 1)] * len(initial_weights)

result = minimize(get_ensemble_rmse, initial_weights, method='SLSQP', bounds=bounds, constraints=constraints)

best_weights = result.x
best_ensemble_rmse = result.fun

print("\n" + "="*50)
print("             OPTIMAL 3-MODEL BLEND SEARCH COMPLETE")
print("="*50)
print(f"Lowest Ensemble RMSE achieved: ${best_ensemble_rmse:,.2f}")
print(f"Optimal Weight for XGBoost  : {best_weights[0]:.4f}")
print(f"Optimal Weight for CatBoost : {best_weights[1]:.4f}")
print(f"Optimal Weight for NeuralNet: {best_weights[2]:.4f}")
print("="*50)


--- Searching for the optimal 3-model blend weights ---

             OPTIMAL 3-MODEL BLEND SEARCH COMPLETE
Lowest Ensemble RMSE achieved: $96,778.10
Optimal Weight for XGBoost  : 0.4100
Optimal Weight for CatBoost : 0.4779
Optimal Weight for NeuralNet: 0.1121


In [13]:
# =======================================================================================
#
# BLOCK 5: THE FULL SUPER-ENSEMBLE PIPELINE
#
# =======================================================================================

# --- PART A: Create the Super-Ensemble Mean ---
print("\n--- PART A: Creating the super-ensemble mean predictions with optimal weights ---")
oof_ensemble_mean = (oof_xgb_preds * best_weights[0] + 
                     oof_cb_preds * best_weights[1] + 
                     oof_nn_preds * best_weights[2])
test_ensemble_mean = (test_xgb_preds * best_weights[0] + 
                      test_cb_preds * best_weights[1] + 
                      test_nn_preds * best_weights[2])

# --- PART B: Tune a New Error Model for the Super-Ensemble ---
print("\n--- PART B: Tuning a new error model for the super-ensemble's errors ---")
error_target_ensemble = np.abs(y_true - oof_ensemble_mean)
X_for_error_ensemble = X.copy()
X_for_error_ensemble['mean_pred_oof'] = oof_ensemble_mean

N_OPTUNA_TRIALS = 75

def create_error_objective(X_features, y_error):
    X_train, X_val, y_train, y_val = train_test_split(X_features, y_error, test_size=0.25, random_state=RANDOM_STATE)
    def objective(trial):
        params = {
            'objective': 'reg:squarederror','eval_metric': 'rmse','tree_method': 'hist',
            'eta': trial.suggest_float('eta', 0.01, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 4, 10),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'lambda': trial.suggest_float('lambda', 1e-8, 10.0, log=True),
            'alpha': trial.suggest_float('alpha', 1e-8, 10.0, log=True),
            'seed': RANDOM_STATE, 'n_jobs': -1
        }
        model = xgb.XGBRegressor(**params, n_estimators=2000, early_stopping_rounds=50)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        return np.sqrt(mean_squared_error(y_val, model.predict(X_val)))
    return objective

study_error_ensemble = optuna.create_study(direction='minimize')
study_error_ensemble.optimize(create_error_objective(X_for_error_ensemble, error_target_ensemble), n_trials=N_OPTUNA_TRIALS)
best_params_error_ensemble = study_error_ensemble.best_params
print(f"\nOptimal Error Model Params Found (RMSE: ${study_error_ensemble.best_value:,.2f})")

# --- PART C: K-Fold Train the Tuned Error Model ---
print("\n--- PART C: K-Fold training the new error model ---")
final_params_error_ensemble = best_params_error_ensemble.copy()
final_params_error_ensemble.update({'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'tree_method': 'hist', 'random_state': RANDOM_STATE, 'n_jobs': -1})
oof_error_preds_ensemble = np.zeros(len(X))
test_error_preds_ensemble = np.zeros(len(X_test))
X_test_for_error_ensemble = X_test.copy()
X_test_for_error_ensemble['mean_pred_oof'] = test_ensemble_mean
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
grade_for_stratify = pd.read_csv(DATA_PATH + 'dataset.csv')['grade']

for fold, (train_idx, val_idx) in enumerate(skf.split(X_for_error_ensemble, grade_for_stratify)):
    print(f"  Training Super-Ensemble Error Model - Fold {fold+1}/{N_SPLITS}...")
    X_train, X_val = X_for_error_ensemble.iloc[train_idx], X_for_error_ensemble.iloc[val_idx]
    y_train, y_val = error_target_ensemble.iloc[train_idx], error_target_ensemble.iloc[val_idx]
    model = xgb.XGBRegressor(**final_params_error_ensemble, n_estimators=2000, early_stopping_rounds=100)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    oof_error_preds_ensemble[val_idx] = model.predict(X_val)
    test_error_preds_ensemble += model.predict(X_test_for_error_ensemble) / N_SPLITS

# --- PART D: Final Calibration and Showdown ---
print("\n--- PART D: Calibrating the final interval and getting the score ---")
OLD_BEST_SCORE = 297707.05 # Your new best score to beat
oof_error_final_ensemble = np.clip(oof_error_preds_ensemble, 0, None)
best_score_ensemble = float('inf')
best_a_ensemble, best_b_ensemble = 1.0, 1.0
for a in np.arange(1.90, 2.31, 0.01):
    for b in np.arange(2.10, 2.51, 0.01):
        low = oof_ensemble_mean - oof_error_final_ensemble * a
        high = oof_ensemble_mean + oof_error_final_ensemble * b
        score = winkler_score(y_true, low, high, alpha=COMPETITION_ALPHA)
        if score < best_score_ensemble:
            best_score_ensemble = score
            best_a_ensemble, best_b_ensemble = a, b

print("\n" + "="*60)
print("             THE SUPER-ENSEMBLE PIPELINE: FINAL SHOWDOWN")
print("="*60)
print(f"Previous Best Score (2-Model Blend) : {OLD_BEST_SCORE:,.2f}")
print(f"New SUPER-ENSEMBLE Pipeline Score   : {best_score_ensemble:,.2f}")
print(f"  (Using optimal multipliers a={best_a_ensemble:.2f}, b={best_b_ensemble:.2f})")

if best_score_ensemble < OLD_BEST_SCORE:
    print("\nCONCLUSION: ABSOLUTE VICTORY! The 3-model super-ensemble is the definitive champion!")
else:
    print("\nCONCLUSION: So close! The 2-model blend remains the champion for now.")

[I 2025-07-16 21:18:33,509] A new study created in memory with name: no-name-77e9237a-b3f7-4338-9b52-8423d82a22c7



--- PART A: Creating the super-ensemble mean predictions with optimal weights ---

--- PART B: Tuning a new error model for the super-ensemble's errors ---


[I 2025-07-16 21:18:38,022] Trial 0 finished with value: 62524.95227898371 and parameters: {'eta': 0.04401664693983837, 'max_depth': 10, 'subsample': 0.6617788919257495, 'colsample_bytree': 0.5989971213578074, 'lambda': 0.0005742731704477911, 'alpha': 9.561831125155212e-07}. Best is trial 0 with value: 62524.95227898371.
[I 2025-07-16 21:18:43,697] Trial 1 finished with value: 62496.383293668696 and parameters: {'eta': 0.018907679696296267, 'max_depth': 4, 'subsample': 0.8904835757080507, 'colsample_bytree': 0.7159922830135126, 'lambda': 2.792411980319109e-07, 'alpha': 0.5433158977170748}. Best is trial 1 with value: 62496.383293668696.
[I 2025-07-16 21:18:46,231] Trial 2 finished with value: 62190.5330622321 and parameters: {'eta': 0.054481906673529715, 'max_depth': 8, 'subsample': 0.698353672188173, 'colsample_bytree': 0.8854360964137469, 'lambda': 4.2309670324003735e-08, 'alpha': 1.750516783117311e-08}. Best is trial 2 with value: 62190.5330622321.
[I 2025-07-16 21:18:50,103] Trial 


Optimal Error Model Params Found (RMSE: $61,739.16)

--- PART C: K-Fold training the new error model ---
  Training Super-Ensemble Error Model - Fold 1/5...
  Training Super-Ensemble Error Model - Fold 2/5...
  Training Super-Ensemble Error Model - Fold 3/5...
  Training Super-Ensemble Error Model - Fold 4/5...
  Training Super-Ensemble Error Model - Fold 5/5...

--- PART D: Calibrating the final interval and getting the score ---

             THE SUPER-ENSEMBLE PIPELINE: FINAL SHOWDOWN
Previous Best Score (2-Model Blend) : 297,707.05
New SUPER-ENSEMBLE Pipeline Score   : 297,175.84
  (Using optimal multipliers a=1.93, b=2.19)

CONCLUSION: ABSOLUTE VICTORY! The 3-model super-ensemble is the definitive champion!


In [14]:
# =======================================================================================
#
# BLOCK 8: STAGE 5 - CREATE FINAL SUBMISSION (CORRECTED)
#
# =======================================================================================
OLD_BEST_SCORE = 301999
if best_score_ensemble < OLD_BEST_SCORE:
    print("\n--- Creating the new champion submission file... ---")
    
    # --- THE FIX IS HERE ---
    # Ignore any previous 'test_ids' variable.
    # Load the correct, original IDs directly from the test.csv file.
    # We use `usecols` to make this very fast and memory-efficient.
    print("Loading original IDs from test.csv...")
    correct_test_ids = pd.read_csv(DATA_PATH + 'test.csv', usecols=['id'])['id']
    
    # The rest of the prediction calculations are correct as they are already in the right order.
    test_error_final_ensemble = np.clip(test_error_preds_ensemble, 0, None)
    final_lower = test_ensemble_mean - test_error_final_ensemble * best_a_ensemble
    final_upper = test_ensemble_mean + test_error_final_ensemble * best_b_ensemble
    final_upper = np.maximum(final_lower, final_upper)
    
    # Create the submission DataFrame using the CORRECT IDs.
    submission_df = pd.DataFrame({'id': correct_test_ids, 'pi_lower': final_lower, 'pi_upper': final_upper})
    submission_filename = 'submission_ensemble_pipeline_v1_CORRECT_IDS.csv'
    submission_df.to_csv(submission_filename, index=False)

    print(f"\n'{submission_filename}' created successfully! Good luck on the leaderboard!")
    display(submission_df.head())
else:
    print("\nNew submission file not created as the score was not an improvement.")


--- Creating the new champion submission file... ---
Loading original IDs from test.csv...

'submission_ensemble_pipeline_v1_CORRECT_IDS.csv' created successfully! Good luck on the leaderboard!


,id,pi_lower,pi_upper
0,200000,834949.166602,1.056976e+06
1,200001,580183.414884,7.847530e+05
2,200002,450826.887000,6.455075e+05
3,200003,287414.684845,4.199714e+05
4,200004,281946.878703,7.749663e+05
